In [37]:
from langgraph.graph import StateGraph , START , END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict , Literal
from dotenv import load_dotenv
from pydantic import BaseModel , Field  
from langchain_core.messages import HumanMessage , SystemMessage

In [38]:
generator_llm = ChatGoogleGenerativeAI(model='gemini-3.5-flash')
evaluator_llm = ChatGoogleGenerativeAI(model='gemini-3.5-flash')
optimizer_llm = ChatGoogleGenerativeAI(model='gemini-3.5-flash')

In [39]:
class TweetEvaluation(BaseModel):
    evaluation: Literal["approved", "needs_improvement"] = Field(..., description="Final evaluation result.")
    feedback: str = Field(..., description="feedback for the tweet.")

structured_evaluator_llm = evaluator_llm.with_structured_output(TweetEvaluation)

In [40]:
class TweetState(TypedDict):
    topic:str
    tweet:str
    evaluation: Literal['approved', 'need_improvement']
    feedback: str
    iteration: int
    max_iteration: int

In [41]:
graph = StateGraph(TweetState)

In [42]:
def generate_tweet(state:TweetState):
    messages = [
        SystemMessage(content="You are a funny and clever Twitter/X influencer."),
        HumanMessage(content=f"""
Write a short, original, and hilarious tweet on the topic: "{state['topic']}".

Rules:
- Do NOT use question-answer format.
- Max 280 characters.
- Use observational humor, irony, sarcasm, or cultural references.
- Think in meme logic, punchlines, or relatable takes.
- Use simple, day to day english
""")
    ]

    response=generator_llm.invoke(messages).content
    return {'tweet':response}

In [43]:
def evaluate_tweet(state:TweetState):
    messages = [
    SystemMessage(content="You are a ruthless, no-laugh-given Twitter critic. You evaluate tweets based on humor, originality, virality, and tweet format."),
    HumanMessage(content=f"""
Evaluate the following tweet:

Tweet: "{state['tweet']}"

Use the criteria below to evaluate the tweet:

1. Originality – Is this fresh, or have you seen it a hundred times before?  
2. Humor – Did it genuinely make you smile, laugh, or chuckle?  
3. Punchiness – Is it short, sharp, and scroll-stopping?  
4. Virality Potential – Would people retweet or share it?  
5. Format – Is it a well-formed tweet (not a setup-punchline joke, not a Q&A joke, and under 280 characters)?

Auto-reject if:
- It's written in question-answer format (e.g., "Why did..." or "What happens when...")
- It exceeds 280 characters
- It reads like a traditional setup-punchline joke
- Dont end with generic, throwaway, or deflating lines that weaken the humor (e.g., “Masterpieces of the auntie-uncle universe” or vague summaries)

### Respond ONLY in structured format:
- evaluation: "approved" or "needs_improvement"  
- feedback: One paragraph explaining the strengths and weaknesses 
""")
]

    response=structured_evaluator_llm.invoke(messages)
    return {'evaluation':response.evaluation, 'feedback': response.feedback, 'feedback_history': [response.feedback]}

In [44]:
def optimize_tweet(state: TweetState):

    messages = [
        SystemMessage(content="You punch up tweets for virality and humor based on given feedback."),
        HumanMessage(content=f"""
Improve the tweet based on this feedback:
"{state['feedback']}"

Topic: "{state['topic']}"
Original Tweet:
{state['tweet']}

Re-write it as a short, viral-worthy tweet. Avoid Q&A style and stay under 280 characters.
""")
    ]

    response = optimizer_llm.invoke(messages).content
    iteration = state['iteration'] + 1

    return {'tweet': response, 'iteration': iteration, 'tweet_history': [response]}

In [45]:
def route_evaluation(state:TweetState):
    if state['evaluation']== 'approved' or state['iteration']>= state['max_iteration']:
        return 'approved'
    else:
        return 'need_improvement'

In [46]:
graph.add_node('generate' , generate_tweet)
graph.add_node('evaluate' , evaluate_tweet)
graph.add_node('optimize' , optimize_tweet)



In [47]:
graph.add_edge(START , 'generate')
graph.add_edge('generate' , 'evaluate')
graph.add_conditional_edges('evaluate' , route_evaluation , {'approved':END , 'need_improvement': 'optimize'})

graph.add_edge('optimize' ,'evaluate')

workflow = graph.compile()

In [49]:
initial_state = {
    "topic": "Indian Railways",
    "iteration": 1,
    "max_iteration": 5
}
workflow.invoke(initial_state)

{'topic': 'Indian Railways',
 'tweet': [{'type': 'text',
   'text': 'Side Upper is the real penthouse of Indian Railways. Yes, you have the vertical clearance of a pizza box, but you get a private window and zero obligation to participate in the compartment’s impromptu Lok Sabha debate.',
   'extras': {'signature': 'EsQlCsElARFNMg+RU/dAg+0WGLH/ZzA1Va64U4YjFXHFerffrKFfMUso7PPGLbytSMFxN0e+mOSVTpgAZQly9X+sWgR49A2k1BE1IO4y5PAG/XEteMAnSC5MeFp+vaMXA4BUpgOA/Db8Ulj2nHpHBOsFHtLPWYZ/fOrQYKQ+h/uRYaqvB0T+MkaNYUe9+tfM+gQEbfesc+gqf0owdD6jhUr9I2vh2Ti7wpD0caBxl65ygRP6oobNLY4mdRnV75x35e9dh16Jvt6sy2Sgq/w3kgen2YywWvNlr9VsaxCJYcXb3KSbF1LmPkJXlgdW76vo3O5Gdcp9/K5P80Bh1Q/QAF6Nv1LmP6klHeLv6yv4vqWDmLZ+Bnd4Gz4FQQzaDw0CwCa3DN5VFwGbzhL3eijeDL3tFc6Jn3oTeQJh7D/j14qmOCJdePaaaH5I3rSrEDWbjDTMAFHf75DwHoisqlvPDfvSrnPkVaJMK6bGrAwcMahwgVh2KZNopCAReuYHGquAEHwD6XbID7GcHKmbE02NlOOgzfiR0STjT4sM4cRB563olLr4YnNF1jKoByoCVYpi5s1MzVnBpyy16BvApnX7Zz4zH+gBKdrdHwcnUpwQYW76Kv5TgHT6hPmgL8DjEh3EH6nm9hT8j3EhIxoTFAWE8g28ONLUL620lO8zbWxCY4